# Load and inspect a Rover data-version-7 capture

This notebook opens a Rover V7 Zarr directly. It preserves and inspects the direct-USB gain/RSSI metadata instead of converting the capture to V5.

Set `V7_ZARR` below to the completed `.zarr` capture. The matching `.yaml` file must be beside it. Run the cells in order and close the LMDB store with the final cell.

In [ ]:
import os
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import yaml

from spf.dataset.v7_data import v7rx_keys
from spf.rf import get_phase_diff
from spf.scripts.zarr_utils import zarr_open_from_lmdb_store

# Either edit this path or start Jupyter with SPF_V7_ZARR set.
V7_ZARR = Path(os.environ.get("SPF_V7_ZARR", "/path/to/rover_capture.zarr"))

# Optional expectations. Leave as None to derive them from the capture.
EXPECTED_RECEIVERS = None
EXPECTED_FRAMES = None

V7_ZARR

## Open and validate the V7 contract

The sidecar YAML identifies the data version and runtime-resolved receiver configuration. The Zarr root attribute identifies the V7 radio-metadata schema.

In [ ]:
if not V7_ZARR.is_dir():
    raise FileNotFoundError(f"Set V7_ZARR to a capture directory: {V7_ZARR}")

yaml_path = V7_ZARR.with_suffix(".yaml")
if not yaml_path.is_file():
    raise FileNotFoundError(f"Missing capture sidecar: {yaml_path}")

config = yaml.safe_load(yaml_path.read_text())
if config.get("data-version") != 7:
    raise ValueError(f"Expected data-version 7, got {config.get('data-version')!r}")

z = zarr_open_from_lmdb_store(str(V7_ZARR), readahead=True, mode="r")
if z.attrs.get("radio_metadata_schema_version") != 2:
    z.store.close()
    raise ValueError(
        "Capture is missing radio_metadata_schema_version=2; it is not the expected V7 schema"
    )

receiver_names = sorted(z["receivers"].group_keys(), key=lambda name: int(name[1:]))
if EXPECTED_RECEIVERS is not None and len(receiver_names) != EXPECTED_RECEIVERS:
    z.store.close()
    raise ValueError(f"Expected {EXPECTED_RECEIVERS} receivers, found {receiver_names}")

print(f"Capture: {V7_ZARR}")
print(f"Receivers: {receiver_names}")
print(f"Metadata schema: {z.attrs['radio_metadata_schema_version']}")
print(f"SDR identity schema: {z.attrs.get('sdr_identity_version', 'missing')}")

In [ ]:
print(z.tree())

required_keys = set(v7rx_keys())
for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    missing = sorted(required_keys - set(receiver.array_keys()))
    if missing:
        raise ValueError(f"{receiver_name} is missing V7 fields: {missing}")
    if receiver.signal_matrix.ndim != 3 or receiver.signal_matrix.shape[1] != 2:
        raise ValueError(
            f"{receiver_name} signal_matrix has unexpected shape {receiver.signal_matrix.shape}"
        )
    if EXPECTED_FRAMES is not None and receiver.signal_matrix.shape[0] != EXPECTED_FRAMES:
        raise ValueError(
            f"{receiver_name}: expected {EXPECTED_FRAMES} frames, "
            f"found {receiver.signal_matrix.shape[0]}"
        )

print("V7 field and signal-shape checks passed")

## Capture overview and valid rows

Preallocated but unwritten rows have a zero `system_timestamp`. All subsequent plots select only written rows.

In [ ]:
valid_masks = {}
overview = []

for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    timestamps = receiver.system_timestamp[:]
    valid = timestamps > 0
    valid_masks[receiver_name] = valid
    valid_timestamps = timestamps[valid]
    intervals = np.diff(valid_timestamps)
    overview.append(
        {
            "receiver": receiver_name,
            "serial": receiver.attrs.get("sdr_serial"),
            "usb_path": receiver.attrs.get("usb_port_path"),
            "allocated_frames": int(timestamps.size),
            "written_frames": int(valid.sum()),
            "buffer_size": int(receiver.signal_matrix.shape[-1]),
            "median_frame_hz": (
                float(1.0 / np.median(intervals)) if intervals.size else np.nan
            ),
        }
    )

overview

## Inspect one IQ buffer from both physical receivers

Choose a written `BUFFER_IDX`. The five aligned panels use a consistent color for each physical-radio/RX-channel pair:

1. Gain snapshots at the start and end of the buffer.
2. RSSI snapshots at the start and end of the buffer.
3. Raw complex IQ, with I solid and Q dotted.
4. Wrapped phase of each individual RX channel.
5. Wrapped RX1−RX2 phase difference for each physical radio.

Only the plotting is decimated; the selected IQ buffer is loaded in full. Endpoint metadata is shown as two points and is not interpolated across the buffer.

In [ ]:
BUFFER_IDX = 0
MAX_PLOT_POINTS = 12_000

for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    if not 0 <= BUFFER_IDX < receiver.signal_matrix.shape[0]:
        raise IndexError(
            f"BUFFER_IDX={BUFFER_IDX} is outside {receiver_name}'s allocated range "
            f"[0, {receiver.signal_matrix.shape[0]})"
        )
    if not valid_masks[receiver_name][BUFFER_IDX]:
        raise ValueError(f"{receiver_name} buffer {BUFFER_IDX} was not written")

buffer_sizes = {
    z[f"receivers/{receiver_name}"].signal_matrix.shape[-1]
    for receiver_name in receiver_names
}
if len(buffer_sizes) != 1:
    raise ValueError(f"Receivers have different IQ buffer sizes: {buffer_sizes}")

buffer_size = buffer_sizes.pop()
plot_stride = max(1, int(np.ceil(buffer_size / MAX_PLOT_POINTS)))
plot_samples = np.arange(0, buffer_size, plot_stride)
if plot_samples[-1] != buffer_size - 1:
    plot_samples = np.append(plot_samples, buffer_size - 1)

print(
    f"Buffer {BUFFER_IDX}: {buffer_size:,} samples/channel; "
    f"plotting {plot_samples.size:,} points with stride {plot_stride}"
)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 19), sharex=True)
gain_ax, rssi_ax, raw_ax, phase_ax, phase_diff_ax = axes

series_colors = plt.get_cmap("tab10").colors
selected_buffer_summary = []

for receiver_index, receiver_name in enumerate(receiver_names):
    receiver = z[f"receivers/{receiver_name}"]
    iq = np.asarray(receiver.signal_matrix[BUFFER_IDX])
    gain_start = np.asarray(receiver.gain_db_start[BUFFER_IDX])
    gain_end = np.asarray(receiver.gain_db_end[BUFFER_IDX])
    rssi_start = np.asarray(receiver.rssi_db_start[BUFFER_IDX])
    rssi_end = np.asarray(receiver.rssi_db_end[BUFFER_IDX])
    phase_difference = get_phase_diff(iq)

    for channel_index in range(2):
        color_index = receiver_index * 2 + channel_index
        color = series_colors[color_index % len(series_colors)]
        label = f"{receiver_name} RX{channel_index + 1}"

        # These are endpoint observations only. Do not draw a line that could
        # be mistaken for known gain/RSSI behavior inside the buffer.
        gain_ax.scatter(0, gain_start[channel_index], color=color, marker="o", s=55)
        gain_ax.scatter(
            buffer_size - 1,
            gain_end[channel_index],
            color=color,
            marker="X",
            s=65,
            label=label,
        )
        rssi_ax.scatter(0, rssi_start[channel_index], color=color, marker="o", s=55)
        rssi_ax.scatter(
            buffer_size - 1,
            rssi_end[channel_index],
            color=color,
            marker="X",
            s=65,
            label=label,
        )

        raw_ax.plot(
            plot_samples,
            iq[channel_index, plot_samples].real,
            color=color,
            linewidth=0.7,
            alpha=0.75,
            label=f"{label} I",
        )
        raw_ax.plot(
            plot_samples,
            iq[channel_index, plot_samples].imag,
            color=color,
            linewidth=0.7,
            alpha=0.6,
            linestyle=":",
            label=f"{label} Q",
        )
        phase_ax.plot(
            plot_samples,
            np.angle(iq[channel_index, plot_samples]),
            color=color,
            linewidth=0.6,
            alpha=0.7,
            label=label,
        )

    receiver_color = series_colors[(receiver_index * 2) % len(series_colors)]
    phase_diff_ax.plot(
        plot_samples,
        phase_difference[plot_samples],
        color=receiver_color,
        linewidth=0.7,
        alpha=0.75,
        label=f"{receiver_name} RX1−RX2 samples",
    )
    stored_average = float(receiver.avg_phase_diff[BUFFER_IDX, 0])
    phase_diff_ax.axhline(
        stored_average,
        color=receiver_color,
        linewidth=1.5,
        linestyle="--",
        label=f"{receiver_name} stored average={stored_average:.3f}",
    )

    selected_buffer_summary.append(
        {
            "receiver": receiver_name,
            "serial": receiver.attrs.get("sdr_serial"),
            "gain_valid": bool(receiver.gain_metadata_valid[BUFFER_IDX]),
            "rssi_valid": bool(receiver.rssi_metadata_valid[BUFFER_IDX]),
            "gain_start_db": gain_start.tolist(),
            "gain_end_db": gain_end.tolist(),
            "rssi_start_db": rssi_start.tolist(),
            "rssi_end_db": rssi_end.tolist(),
            "gain_endpoints_equal": receiver.gain_endpoints_equal[BUFFER_IDX].tolist(),
            "stream_id": int(receiver.stream_id[BUFFER_IDX]),
            "buffer_sequence": int(receiver.buffer_sequence[BUFFER_IDX]),
            "sample_sequence": int(receiver.sample_sequence[BUFFER_IDX]),
        }
    )

gain_ax.set_title("Gain endpoint snapshots: circle=start, X=end")
gain_ax.set_ylabel("Gain (dB)")
rssi_ax.set_title("RSSI endpoint snapshots: circle=start, X=end")
rssi_ax.set_ylabel("RSSI (dB)")
raw_ax.set_title("Raw complex IQ: I=solid, Q=dotted")
raw_ax.set_ylabel("ADC value")
phase_ax.set_title("Wrapped phase of each RX channel")
phase_ax.set_ylabel("Phase (rad)")
phase_ax.set_ylim(-np.pi, np.pi)
phase_diff_ax.set_title("Wrapped RX1−RX2 phase difference")
phase_diff_ax.set_ylabel("Phase difference (rad)")
phase_diff_ax.set_xlabel("Sample index within selected buffer")
phase_diff_ax.set_ylim(-np.pi, np.pi)

for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right", ncol=2, fontsize=8)

fig.suptitle(f"V7 buffer {BUFFER_IDX}: both physical receivers", fontsize=16)
plt.tight_layout()
plt.show()

selected_buffer_summary

## Legacy phase field

`avg_phase_diff[:, 0]` remains available in V7, so existing phase plots require no schema translation.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    valid = valid_masks[receiver_name]
    frame_indexes = np.flatnonzero(valid)
    phase = receiver.avg_phase_diff[:][valid, 0]
    ax.scatter(frame_indexes, phase, s=4, label=receiver_name)

ax.set(title="Average RX1−RX2 phase difference", xlabel="Frame", ylabel="Phase (rad)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## Gain/RSSI metadata quality and stream continuity

V7 records gain and RSSI in dB near both frame endpoints. Endpoint equality means only that the two snapshots agree; it does not prove that no transition occurred inside the frame.

In [ ]:
quality = []

for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    valid = valid_masks[receiver_name]
    frame_indexes = np.flatnonzero(valid)

    gain_valid = receiver.gain_metadata_valid[:][valid]
    rssi_valid = receiver.rssi_metadata_valid[:][valid]
    endpoints_equal = receiver.gain_endpoints_equal[:][valid]
    stream_ids = receiver.stream_id[:][valid]
    buffer_sequences = receiver.buffer_sequence[:][valid]
    sample_sequences = receiver.sample_sequence[:][valid]

    same_stream = stream_ids[1:] == stream_ids[:-1]
    buffer_gaps = frame_indexes[1:][
        same_stream & (buffer_sequences[1:] != buffer_sequences[:-1] + 1)
    ]
    expected_sample_step = receiver.signal_matrix.shape[-1]
    sample_gaps = frame_indexes[1:][
        same_stream & (sample_sequences[1:] != sample_sequences[:-1] + expected_sample_step)
    ]

    quality.append(
        {
            "receiver": receiver_name,
            "gain_valid": f"{int(gain_valid.sum())}/{gain_valid.size}",
            "rssi_valid": f"{int(rssi_valid.sum())}/{rssi_valid.size}",
            "endpoint_changed_frames": int((~endpoints_equal.all(axis=1)).sum()),
            "unique_streams": int(np.unique(stream_ids).size),
            "buffer_gap_frames": buffer_gaps.tolist(),
            "sample_gap_frames": sample_gaps.tolist(),
        }
    )

quality

In [ ]:
# V7 preserves the legacy fields for old consumers. They must equal the
# end-of-frame metadata values, but the explicit V7 names are preferred.
for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    valid = valid_masks[receiver_name]
    if not np.allclose(receiver.gains[:][valid], receiver.gain_db_end[:][valid], equal_nan=True):
        raise ValueError(f"{receiver_name}: legacy gains differ from gain_db_end")
    if not np.allclose(receiver.rssis[:][valid], receiver.rssi_db_end[:][valid], equal_nan=True):
        raise ValueError(f"{receiver_name}: legacy RSSIs differ from rssi_db_end")

print("Legacy gain/RSSI compatibility checks passed")

## Plot endpoint gain and RSSI

Solid lines are the start snapshots and dotted lines are the end snapshots. RX1 and RX2 are plotted separately for each physical radio.

In [ ]:
fig, axes = plt.subplots(len(receiver_names), 2, figsize=(14, 4 * len(receiver_names)), squeeze=False)

for row, receiver_name in enumerate(receiver_names):
    receiver = z[f"receivers/{receiver_name}"]
    valid = valid_masks[receiver_name]
    frames = np.flatnonzero(valid)
    gain_start = receiver.gain_db_start[:][valid]
    gain_end = receiver.gain_db_end[:][valid]
    for channel in range(2):
        ax = axes[row, channel]
        ax.plot(frames, gain_start[:, channel], label="start", linewidth=1)
        ax.plot(frames, gain_end[:, channel], label="end", linewidth=1, linestyle=":")
        ax.set(
            title=f"{receiver_name} RX{channel + 1} gain",
            xlabel="Frame",
            ylabel="Gain (dB)",
        )
        ax.grid(alpha=0.2)
        ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(receiver_names), 2, figsize=(14, 4 * len(receiver_names)), squeeze=False)

for row, receiver_name in enumerate(receiver_names):
    receiver = z[f"receivers/{receiver_name}"]
    valid = valid_masks[receiver_name]
    frames = np.flatnonzero(valid)
    rssi_start = receiver.rssi_db_start[:][valid]
    rssi_end = receiver.rssi_db_end[:][valid]
    for channel in range(2):
        ax = axes[row, channel]
        ax.plot(frames, rssi_start[:, channel], label="start", linewidth=1)
        ax.plot(frames, rssi_end[:, channel], label="end", linewidth=1, linestyle=":")
        ax.set(
            title=f"{receiver_name} RX{channel + 1} RSSI",
            xlabel="Frame",
            ylabel="RSSI (dB)",
        )
        ax.grid(alpha=0.2)
        ax.legend()

plt.tight_layout()
plt.show()

## IQ power and rover GPS

`iq_power_dbfs` is computed from the IQ payload and is frame-aligned. It is post-gain digital power, not calibrated antenna-input dBm.

In [ ]:
fig, (power_ax, gps_ax) = plt.subplots(1, 2, figsize=(15, 5))

for receiver_name in receiver_names:
    receiver = z[f"receivers/{receiver_name}"]
    valid = valid_masks[receiver_name]
    frames = np.flatnonzero(valid)
    power = receiver.iq_power_dbfs[:][valid]
    power_ax.plot(frames, power[:, 0], label=f"{receiver_name} RX1", linewidth=1)
    power_ax.plot(frames, power[:, 1], label=f"{receiver_name} RX2", linewidth=1)

    lat = receiver.gps_lat[:][valid]
    lon = receiver.gps_long[:][valid]
    gps_valid = np.isfinite(lat) & np.isfinite(lon) & (lat != 0) & (lon != 0)
    gps_ax.scatter(lon[gps_valid], lat[gps_valid], s=5, label=receiver_name)

power_ax.set(title="Frame-aligned IQ power", xlabel="Frame", ylabel="Power (dBFS)")
power_ax.grid(alpha=0.2)
power_ax.legend()
gps_ax.set(title="Receiver GPS track", xlabel="Longitude", ylabel="Latitude")
gps_ax.grid(alpha=0.2)
gps_ax.legend()
plt.tight_layout()
plt.show()

## Per-radio firmware and hardware provenance

In [ ]:
print("Root attributes")
pprint(dict(z.attrs))

for receiver_name in receiver_names:
    print(f"\n{receiver_name} attributes")
    pprint(dict(z[f"receivers/{receiver_name}"].attrs))

## Optional legacy segmentation compatibility

`v5spfdataset` is not a native V7 loader. If an older segmentation workflow is required, it can read the legacy subset with `v4=True`. It will ignore the V7 metadata and synthesize unavailable V5 position fields as zeros, so do not use its derived ground-truth geometry for a raw Rover V7 capture.

Generating segmentation for production-size IQ frames can be expensive. This cell is disabled by default.

In [ ]:
RUN_LEGACY_SEGMENTATION = False
PRECOMPUTE_CACHE = Path("/tmp/spf-segmentation-cache")

legacy_ds = None
if RUN_LEGACY_SEGMENTATION:
    from spf.dataset.spf_dataset import v5spfdataset

    PRECOMPUTE_CACHE.mkdir(parents=True, exist_ok=True)
    legacy_ds = v5spfdataset(
        str(V7_ZARR),
        nthetas=config.get("n-thetas", 65),
        ignore_qc=True,
        precompute_cache=str(PRECOMPUTE_CACHE),
        gpu=True,
        snapshots_per_session=1,
        n_parallel=4,
        paired=True,
        segment_if_not_exist=True,
        v4=True,
    )
    print(f"Loaded {len(legacy_ds)} legacy-compatible sessions")

## Close the stores

Run this before changing `V7_ZARR` or ending the notebook.

In [ ]:
if legacy_ds is not None:
    legacy_ds.close()
    legacy_ds = None

z.store.close()
z = None
print("Closed V7 capture stores")